# 51. Online DPO | 在线 DPO
**难度：** Hard | **环境：** CPU-first | **标签：** `后训练对齐`, `DPO`, `在线优化` | **目标人群：** 项目决策练习者

---

## 本节导读

在线 DPO 关心的不是“新样本来了就更新”，而是反馈是否仍代表当前策略、更新后是否真的改善，以及发现退化时能否回到可验证的版本。Part 02 · 50 建立数据和评测契约，Part 02 · 17 解释离线偏好目标；本节接着处理新反馈持续进入时的版本、新鲜度与回滚问题。

在线 DPO 消费 Part 02 · 50 产出的合格偏好对，以及轨迹生命周期页产出的策略新鲜度状态。本节不重新定义通用轨迹准入，而是把成对反馈组织成固定窗口，执行有限 DPO 更新，并依据独立评测作出采用、延迟或回滚决策。

**关键词：** `online feedback`, `policy freshness`, `evaluation isolation`, `rollback`

---


## 前置阅读

**导语：** 先建立离线偏好目标、评测协议和策略版本语义，再观察持续反馈如何改变更新准入。

- [Part 02 · 50 偏好数据与对齐评测](./50_Preference_Data_and_Evaluation.ipynb)
- [Part 02 · 17 DPO 损失](./17_DPO_Loss_Tutorial.ipynb)
- [Part 02 · PT-ROLLOUT-LIFECYCLE Rollout 与策略新鲜度](./PT-ROLLOUT-LIFECYCLE_Rollout_and_Policy_Freshness.ipynb)



### Step 1：在线反馈如何进入更新闭环

在线偏好优化把一次离线训练扩展为持续循环。新反馈先形成同一任务下的优选 / 非优选回答对；来自工具执行的轨迹，需先根据终态验证结果构造成对偏好。通过身份、版本、风险与切分检查的样本进入固定窗口，完成有限更新后再接受独立评测。

| 阶段 | 核心关系 | 作用 |
|:---|:---|:---|
| 成对构造 | 同一任务下的优选与非优选回答具有明确选择依据 | 把反馈转换为 DPO 可消费的数据对象 |
| 反馈准入 | 样本身份、产生策略、来源与数据切分相互对应 | 判断偏好对能否进入当前窗口 |
| 有限更新 | 一轮更新只消费边界明确的反馈窗口 | 隔离数据变化与策略变化 |
| 独立复核 | 更新候选与固定基线接受同口径评测 | 防止训练反馈替代最终证据 |
| 采用与恢复 | 收益判断与可恢复版本同时存在 | 在退化时停止或回到稳定版本 |

![在线反馈经过筛选、有限更新和独立评测进入采用或回滚](../docs/public/02_PyTorch_Algorithms/51_online_dpo_lifecycle.svg)


### Step 2：固定反馈窗口如何隔离数据变化与策略变化

每条偏好对先携带轨迹生命周期页给出的策略新鲜度状态；本节再检查重复消费、偏好对完整性、风险状态和评测切分。窗口固定最大样本数、排序和样本身份，使一次 DPO 更新对应唯一的数据边界。

| 判断维度 | 要回答的问题 | 风险响应 |
|:---|:---|:---|
| 身份完整性 | 同一反馈能否唯一识别并避免重复消费 | 去重、补证或拒绝 |
| 策略新鲜度状态 | 上游是否判定反馈仍可用于当前策略 | 接纳、复核或重新采样 |
| 风险状态 | 样本是否需要人工或规则复核 | 阻止自动进入更新 |
| 切分隔离 | 反馈是否污染独立评测 | 移出更新池并恢复隔离 |
| 偏好对完整性 | chosen 与 rejected 是否来自同一任务且内容不同 | 补证或拒绝 |
| 窗口边界 | 本轮更新消费哪些 pair ID | 固定顺序、容量和样本清单 |

![策略版本差与身份、风险、切分检查共同决定反馈准入](../docs/public/02_PyTorch_Algorithms/51_policy_freshness_gate.svg)


### Step 3：用独立评测检查更新收益与退化

在线更新完成后，候选必须与固定基线在独立评测条件下比较。质量、稳定性、安全与更新成本需要分开观察；工具任务还要检查任务终态和工具调用是否退化。任何安全硬门槛失败都应优先于平均质量收益，局部失败集中也不能被总体均值掩盖。

| 证据维度 | 解释 | 决策作用 |
|:---|:---|:---|
| 成对质量变化 | 候选相对基线是否获得稳定收益 | 判断是否值得继续观察 |
| 跨窗口稳定性 | 收益是否在重复评测和后续窗口中保持 | 防止短期波动被当成改进 |
| 安全与失败分布 | 是否出现不可接受或集中的退化 | 触发延迟、停止或回滚 |
| 更新成本 | 反馈获取、参数更新和复测消耗多少资源 | 判断收益是否值得持续投入 |
| 任务终态（可选） | 轨迹是否真正完成任务 | 防止文本偏好掩盖执行失败 |


### Step 4：实现反馈窗口与回滚决策

题目区先判断单个偏好对的准入原因，再形成边界固定的更新窗口，最后结合独立评测与恢复条件给出 `adopt / defer / rollback / halt`。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|:---|:---|:---|:---|
| TODO 1：判断偏好对准入<br>`classify_preference_pair` | 检查成对语义、切分、风险和策略版本 | 每个样本只返回一个首要拒绝原因 | 完整、同答、污染、风险、新鲜度边界 |
| TODO 2：形成更新窗口<br>`build_preference_window` | 去重并固定本轮消费的偏好对 | pair ID 唯一、顺序确定且不超过窗口容量 | 样本清单、拒绝计数与来源覆盖 |
| TODO 3：选择采用或回滚<br>`recommend_online_dpo` | 联合更新门槛与回滚准备形成决策 | 安全退化先检查恢复路径，软门槛不能触发 adopt | `adopt / defer / rollback / halt` |

## 测试

运行下方 CPU 机制测试，分别检查反馈准入、证据对照和采用或回滚决策。


In [ ]:
from typing import Dict, List


In [ ]:
def classify_preference_pair(
    pair: Dict[str, object], current_policy_version: int, max_policy_lag: int
) -> str:
    """返回 eligible 或一个首要拒绝原因。"""
    # ==========================================
    # TODO 1：判断单个偏好对能否进入在线更新。
    # pair_id = ???  # 偏好对的稳定身份。
    # policy_lag = ???  # 当前策略与数据产生策略的版本差。
    # ==========================================
    raise NotImplementedError("TODO 1：判断偏好对准入")


def build_preference_window(
    feedback_stream: List[Dict[str, object]],
    current_policy_version: int,
    max_policy_lag: int,
    max_pairs: int,
) -> Dict[str, object]:
    """去重、准入并形成顺序稳定的偏好窗口。"""
    # ==========================================
    # TODO 2：形成边界固定的在线偏好窗口。
    # accepted_pairs = ???  # 通过准入且未重复的偏好对。
    # selected_pairs = ???  # 按 pair_id 排序后截取的本轮窗口。
    # ==========================================
    raise NotImplementedError("TODO 2：形成在线偏好窗口")


def recommend_online_dpo(
    offline_run: Dict[str, float],
    online_run: Dict[str, float],
    min_win_rate_gain: float,
    rollback_ready: bool,
    max_stability_drop: float = 0.05,
    max_safety_drop: float = 0.02,
) -> Dict[str, object]:
    """根据更新证据与恢复条件返回采用、延迟、回滚或停止。"""
    # ==========================================
    # TODO 3：根据独立评测和恢复条件决定是否采用候选。
    # comparison = ???  # 分别保留质量、稳定性、安全与成本变化。
    # decision = ???  # adopt、defer、rollback 或 halt。
    # ==========================================
    raise NotImplementedError("TODO 3：形成在线更新与回滚决策")

In [ ]:
def _pair(pair_id, version=10, chosen='good', rejected='bad', split='train', risk=False, source='prod'):
    return {"pair_id": pair_id, "prompt_id": f"prompt-{pair_id}", "chosen": chosen,
            "rejected": rejected, "policy_version": version, "split": split,
            "risk": risk, "source": source}


def test_pair_admission_reasons():
    """成对语义、切分、风险和版本差必须分别产生明确原因。"""
    assert classify_preference_pair(_pair('ok'), 10, 2) == 'eligible'
    assert classify_preference_pair(_pair('same', chosen='x', rejected='x'), 10, 2) == 'invalid_pair'
    assert classify_preference_pair(_pair('eval', split='evaluation'), 10, 2) == 'evaluation_leakage'
    assert classify_preference_pair(_pair('risk', risk=True), 10, 2) == 'risk'
    assert classify_preference_pair(_pair('stale', version=7), 10, 2) == 'stale'
    assert classify_preference_pair(_pair('future', version=11), 10, 2) == 'future'


def test_preference_window_identity_and_capacity():
    """窗口必须去重、稳定排序并遵守最大容量。"""
    stream = [
        _pair('b', source='shadow'), _pair('a'), _pair('b', source='shadow'),
        _pair('stale', version=6), _pair('unsafe', risk=True),
    ]
    report = build_preference_window(stream, 10, 2, max_pairs=2)
    assert report["pair_ids"] == ['a', 'b']
    assert report["rejection_counts"] == {'duplicate': 1, 'risk': 1, 'stale': 1}
    assert report["eligible_sources"] == ['prod', 'shadow']


def test_adopt_defer_rollback_and_halt():
    """安全退化需要恢复路径，软门槛不足只延迟采用。"""
    offline = {"win_rate": 0.52, "stability": 0.82, "safety_score": 0.95, "update_cost": 1.0}
    adopt = {"win_rate": 0.59, "stability": 0.79, "safety_score": 0.94, "update_cost": 1.2}
    unsafe = {**adopt, "safety_score": 0.90}
    assert recommend_online_dpo(offline, adopt, 0.05, rollback_ready=True)["decision"] == "adopt"
    assert recommend_online_dpo(offline, {**adopt, "win_rate": 0.54}, 0.05, rollback_ready=True)["decision"] == "defer"
    assert recommend_online_dpo(offline, unsafe, 0.05, rollback_ready=True)["decision"] == "rollback"
    assert recommend_online_dpo(offline, unsafe, 0.05, rollback_ready=False)["decision"] == "halt"


try:
    test_pair_admission_reasons()
    test_preference_window_identity_and_capacity()
    test_adopt_defer_rollback_and_halt()
    print("✅ 在线反馈窗口、更新对照与回滚准备测试通过。")
except (AttributeError, NameError, TypeError, ValueError, AssertionError) as error:
    raise NotImplementedError("请先完成 TODO 代码！") from error

---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
from typing import Dict, List


def classify_preference_pair(
    pair: Dict[str, object], current_policy_version: int, max_policy_lag: int
) -> str:
    """返回 eligible 或一个首要拒绝原因。"""
    # TODO 1：判断偏好对准入。
    pair_id = str(pair.get('pair_id', ''))
    if not pair_id or not pair.get('prompt_id') or not pair.get('chosen') or not pair.get('rejected'):
        return 'missing_identity'
    if pair['chosen'] == pair['rejected']:
        return 'invalid_pair'
    if pair.get('split') != 'train':
        return 'evaluation_leakage'
    if bool(pair.get('risk', False)):
        return 'risk'
    policy_lag = current_policy_version - int(pair.get('policy_version', -1))
    if policy_lag < 0:
        return 'future'
    if policy_lag > max_policy_lag:
        return 'stale'
    return 'eligible'


def build_preference_window(feedback_stream, current_policy_version, max_policy_lag, max_pairs):
    """去重、准入并形成顺序稳定的偏好窗口。"""
    # TODO 2：形成在线偏好窗口。
    if max_pairs <= 0:
        raise ValueError('max_pairs 必须为正')
    seen_ids, accepted_pairs = set(), []
    rejection_counts = {}
    for pair in feedback_stream:
        pair_id = str(pair.get('pair_id', ''))
        reason = 'duplicate' if pair_id in seen_ids else classify_preference_pair(pair, current_policy_version, max_policy_lag)
        seen_ids.add(pair_id)
        if reason == 'eligible':
            accepted_pairs.append(pair)
        else:
            rejection_counts[reason] = rejection_counts.get(reason, 0) + 1
    selected_pairs = sorted(accepted_pairs, key=lambda item: str(item['pair_id']))[:max_pairs]
    return {'pair_ids': [str(item['pair_id']) for item in selected_pairs],
            'eligible_sources': sorted({str(item.get('source', 'unknown')) for item in selected_pairs}),
            'rejection_counts': dict(sorted(rejection_counts.items()))}


def recommend_online_dpo(
    offline_run: Dict[str, float],
    online_run: Dict[str, float],
    min_win_rate_gain: float,
    rollback_ready: bool,
    max_stability_drop: float = 0.05,
    max_safety_drop: float = 0.02,
) -> Dict[str, object]:
    """根据更新收益、退化门槛与恢复准备形成决策。"""
    # TODO 3：选择采用、延迟或回滚。
    comparison = {
        'win_rate_gain': round(online_run['win_rate'] - offline_run['win_rate'], 6),
        'stability_delta': round(online_run['stability'] - offline_run['stability'], 6),
        'safety_delta': round(online_run['safety_score'] - offline_run['safety_score'], 6),
        'cost_delta': round(online_run['update_cost'] - offline_run['update_cost'], 6),
    }
    if comparison["safety_delta"] < -max_safety_drop:
        decision = "rollback" if rollback_ready else "halt"
        reason = "safety_regression"
    elif comparison["win_rate_gain"] >= min_win_rate_gain and comparison["stability_delta"] >= -max_stability_drop:
        decision = "adopt"
        reason = "quality_and_stability_pass"
    else:
        decision = "defer"
        reason = "soft_gate_not_met"
    return {"decision": decision, "reason": reason, "comparison": comparison, "rollback_ready": rollback_ready}

### 解析

#### TODO 1：判断偏好对准入

- 先验证偏好对本身，再检查训练切分、风险和策略版本；每条数据只保留一个首要拒绝原因。

#### TODO 2：形成在线偏好窗口

- pair ID 去重和稳定排序使窗口可复现，最大容量让一次更新拥有明确的数据边界。

#### TODO 3：选择采用、延迟或回滚

- 安全退化优先触发恢复检查；没有可靠 checkpoint 时停止更新而不是继续采用。

## 相关阅读

- [Part 02 · PT-RL-SYSTEM 采样、训练与奖励系统](./PT-RL-SYSTEM_Rollout_Training_and_Reward_System.ipynb)
- [Part 02 · 84 DPO 偏好优化项目](./84_DPO_Preference_Project.ipynb)
- [Part 02 · 86 DPO 在线 Benchmark](./86_DPO_Online_Benchmark.ipynb)
- [TRL: DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)
